# Generate an Evaluation Dataset for a RAG Agent

In this notebook, you'll create a synthetic evaluation dataset using **NVIDIA NeMo Data Designer**, an open source tool for synthetic data generation. We'll generate test cases for evaluating the **IT Help Desk RAG Agent** from Module 2.

## 1. Setup

First, let's set up our environment.

In [ ]:
# Dependencies are provided by the locked project environment.

from dotenv import load_dotenv
_ = load_dotenv("../../variables.env")
_ = load_dotenv("../../secrets.env")

import os
import json
from pathlib import Path

## 2. Configure DataDesigner

Next, initialize Data Designer with default settings.


In [ ]:
from data_designer.essentials import (
    DataDesigner,
    DataDesignerConfigBuilder,
    LLMTextColumnConfig,
    ModelConfig,
    DataFrameSeedSource,
)

model_configs = [ModelConfig(alias="nvidia-text", model="nvidia/nemotron-3-super-120b-a12b")]

# Initialize Data Designer with default settings
data_designer = DataDesigner()
config_builder = DataDesignerConfigBuilder(model_configs)

print("Completed initialization")

## 3. Load IT Knowledge Base as Seed Data

We’ll use articles from the IT knowledge base as seed data for Data Designer. Seed data is the initial set of examples Data Designer uses to guide what it generates.

By seeding Data Designer with our agent's knowledge base, we'll ensure the evaluation data is grounded in information the agent is expected to know.


In [ ]:
import pandas as pd

# Load all articles from the knowledge base directory
kb_dir = Path("../../data/it-knowledge-base")
kb_articles = []

for kb_file in sorted(kb_dir.glob("*.md")):
    with open(kb_file, 'r') as f:
        content = f.read()
    
    # Extract the article title and create a readable topic name
    topic_name = kb_file.stem.replace('-', '_')
    title = content.split('\n')[0].replace('# ', '').strip()
    
    kb_articles.append({
        "topic": topic_name,
        "title": title,
        "filename": kb_file.name,
        "content": content
    })

# Create seed data DataFrame
seed_data = pd.DataFrame(kb_articles)

# Data Designer requires seed data to be saved as a file for efficient sampling
seed_ref = DataFrameSeedSource(
    df=seed_data
)

print(f"Created seed reference with {len(seed_data)} KB articles")

## 4. Configure Dataset Structure for RAG Evaluation

We'll use Data Designer’s Config Builder to define the structure of our synthetic dataset. Our configuration will include the following components:

- Category (sampled from a list of knowledge base topics)
- IT help desk questions (based on category)
- Context keywords (to evaluate retrieval)
- Ground truth answers (ideal agent answer)

We do this by defining a list of "columns" representing the different fields of data that we want to generate. Using Jinja formatting, we can reference items in other columns in the same "row." This allows us to, for example, generate a question and then generate an answer based on that question.

In [ ]:
# Create config builder with seed data
config_builder.with_seed_dataset(seed_ref)

# Generate question based on the seed article
config_builder.add_column(
    LLMTextColumnConfig(
        name="question",
        model_alias="nvidia-text",
        prompt="""Generate a realistic IT help desk question that a user might ask about the topic {{ topic }} from article {{ title }}.

The question should:
- Sound like a real user would ask it
- Be specific and clear
- Cover common scenarios related to this topic
- Be directly answerable using the information in this KB article
- NOT just restate the article title

Return ONLY the question, nothing else.""",
    )
)

# Generate context keywords for retrieval verification
config_builder.add_column(
    LLMTextColumnConfig(
        name="context_key_words",
        model_alias="nvidia-text",
        prompt="""Based on this IT help desk question: {{ question }}

List 2-3 key words or phrases that would help locate the relevant knowledge base article.

Return as a comma-separated list.
Return ONLY the keywords, nothing else.""",
    )
)

# Generate ground truth answer based on the knowledge base
config_builder.add_column(
    LLMTextColumnConfig(
        name="ground_truth",
        model_alias="nvidia-text",
        prompt="""Based on this IT help desk question: {{ question }}

And this knowledge base article content:
{{ content }}

Generate a concise, accurate answer that directly addresses the question using only information from the knowledge base.

Return ONLY the answer, nothing else.""",
    )
)

# Generate category based on the knowledge base
config_builder.add_column(
    LLMTextColumnConfig(
        name="category",
        model_alias="nvidia-text",
        prompt="""Based on this IT help desk question: {{ question }}

Generate a concise, accurate category label for this question. For multi-word categories, replace all spaces with an underscore character. 

Return ONLY the answer, nothing else.""",
    )
)

print("Dataset configuration built")

## 5. Preview the Data Structure

Let's preview what the generated test cases will look like.

In [ ]:
# Preview a few sample records to see what will be generated
preview = data_designer.preview(config_builder=config_builder, num_records=3)
preview.display_sample_record()


## 6. Generate the Full Dataset

Now let's generate the complete evaluation dataset.

In [ ]:
# We'll only generate a few records for this tutorial
dataset_results = data_designer.create(
    config_builder=config_builder,
    num_records = 10,
    dataset_name="rag_agent_test_cases"
)

# Convert to list of dictionaries for processing
eval_data = dataset_results.load_dataset().to_dict('records')

print(f"Generated {len(eval_data)} test cases")


## QA Step: Human Review of Generated Data

Before saving, let's review a small sample of the generated test cases. Synthetic data can contain errors, odd phrasing, or cases that don't make sense. A quick human review helps catch issues early.

**Take 2-3 minutes to review the samples below and verify they look reasonable.**


In [ ]:
# QA Review: Display a few samples for human validation
print("=" * 80)
print("QA REVIEW: Validate Generated Test Cases")
print("=" * 80)
print("\nReview each sample and check:")
print("  ✓ Does the question sound natural and realistic?")
print("  ✓ Are the keywords relevant to the question?")
print("  ✓ Is the ground truth answer accurate and complete?")
print("  ✓ Would this be a fair test for the RAG agent?\n")

qa_sample_size = min(3, len(eval_data))

for i, record in enumerate(eval_data[:qa_sample_size]):
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}/{qa_sample_size}")
    print(f"{'='*80}")
    
    question = record.get('question', 'N/A')
    keywords = record.get('context_key_words', 'N/A')
    ground_truth = record.get('ground_truth', 'N/A')
    category = record.get('category', 'N/A')
    
    print(f"\n❓ QUESTION:\n   {question}")
    print(f"\n🔑 KEYWORDS:\n   {keywords}")
    print(f"\n✅ GROUND TRUTH:\n   {ground_truth[:750]}{'...' if len(str(ground_truth)) > 750 else ''}")
    print(f"\n📋 CATEGORY:\n   {category}")
    
    print(f"\n📋 QA CHECKLIST:")
    print("   [ ] Question is clear and realistic")
    print("   [ ] Keywords would help find relevant docs")
    print("   [ ] Ground truth accurately answers the question")
    print("   [ ] Category is accurately labeled")

print("\n" + "=" * 80)
print("QA SUMMARY")
print("=" * 80)
print(f"""
Total generated: {len(eval_data)} test cases
Reviewed: {qa_sample_size} samples

If you found issues:
  1. Re-run generation with adjusted prompts
  2. Manually edit problematic cases after saving
  3. Remove low-quality cases from the final dataset

Proceed to save if the samples look good!
""")

## 7. Save the Test Data

Now let's clean up and save the generated data in a format ready for evaluation.

We'll keep only the essential evaluation fields: **question**, **context_key_words**, and **ground_truth**. We'll remove the seed data and reasoning traces to make the output cleaner and easier to review.

In [ ]:
# Set output path
output_path = Path("../../data/evaluation/synthetic_rag_agent_test_cases.json")

# Keep only essential evaluation columns, ignore reasoning traces and seed data
essential_columns = ['question', 'context_key_words', 'ground_truth', 'category']
clean_data = []

for record in eval_data:
    clean_record = {}
    for key, value in record.items():
        if key in essential_columns:
            clean_record[key] = value
    clean_data.append(clean_record)

# Save to JSON file
with open(output_path, 'w') as f:
    json.dump(clean_data, f, indent=2)

print(f"Saved {len(eval_data)} test cases to {output_path}")

## What's Next?

Check out ``/data/evaluation/synthetic_rag_agent_test_cases.json`` to view your generated data.

In the next notebook, you'll use this same process to create an evaluation dataset for your Report Generation agent!